In [2]:
import pandas as pd

threat = pd.read_csv(
    "threat_database.csv"
)

print("Rows:", len(threat))
print("Columns:", len(threat.columns))

print(threat.columns.tolist())

Rows: 4349
Columns: 14
['medicare_id', 'attacked', 'attack_date', 'er_diversion', 'cancel_delay', 'threat_id', 'nearest_neighbor', 'origin_threat_id1', 'origin_threat_id2', 'origin_threat_id3', 'origin_threat_id4', 'origin_threat_id5', 'origin_threat_id6', 'origin_threat_id7']


In [3]:
print("Unique hospitals:",
      threat["medicare_id"].nunique())

print("Unique threats:",
      threat["threat_id"].nunique())

Unique hospitals: 4337
Unique threats: 74


In [4]:
print(
    threat.groupby("threat_id")["medicare_id"]
    .nunique()
    .describe()
)

count    74.000000
mean      2.013514
std       3.054273
min       1.000000
25%       1.000000
50%       1.000000
75%       1.750000
max      22.000000
Name: medicare_id, dtype: float64


In [5]:
duplicates = threat.duplicated(
    subset=["medicare_id", "threat_id"],
    keep=False
)

print(threat[duplicates].sort_values(
    ["threat_id", "medicare_id"]
).head(20))

     medicare_id  attacked attack_date  er_diversion  cancel_delay  threat_id  \
110       370091         1    9/7/2016           0.0           0.0       28.0   
111       370091         1    9/7/2016           0.0           0.0       28.0   
1          10092         1   10/1/2019           1.0           0.0      127.0   
2          10092         1   10/1/2019           1.0           0.0      127.0   
85        310039         1  11/29/2019           0.0           1.0      133.0   
86        310039         1  11/29/2019           0.0           1.0      133.0   
46        180067         1    2/8/2020           0.0           0.0      142.0   
47        180067         1    2/8/2020           0.0           0.0      142.0   
127       390180         1   6/14/2020           0.0           0.0      166.0   
128       390180         1   6/14/2020           0.0           0.0      166.0   
129       390180         1   6/14/2020           0.0           0.0      166.0   
15         50701         1  

In [6]:
medicare = pd.read_csv(
    "Medicare_IP_Hospitals_by_Provider_2024.csv")
threat["CCN"] = (
    threat["medicare_id"]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)
medicare["CCN"] = (
    medicare["Rndrng_Prvdr_CCN"]
    .astype(str)
    .str.strip()
    .str.zfill(6)
)
common = set(threat["CCN"]) & set(medicare["CCN"])

print("THREAT hospitals:",
      threat["CCN"].nunique())

print("Medicare hospitals:",
      medicare["CCN"].nunique())

print("Matched hospitals:",
      len(common))


THREAT hospitals: 4337
Medicare hospitals: 3044
Matched hospitals: 2791


In [7]:
print("Total duplicate rows:",
      duplicates.sum())

print("Unique duplicated hospital-threat pairs:",
      threat.loc[duplicates, ["medicare_id", "threat_id"]]
      .drop_duplicates()
      .shape[0])

print(
    threat.loc[
        duplicates,
        ["medicare_id", "threat_id", "attack_date",
         "attacked", "er_diversion", "cancel_delay"]
    ]
    .sort_values(["threat_id", "medicare_id"])
    .head(30)
)

Total duplicate rows: 20
Unique duplicated hospital-threat pairs: 9
     medicare_id  threat_id attack_date  attacked  er_diversion  cancel_delay
110       370091       28.0    9/7/2016         1           0.0           0.0
111       370091       28.0    9/7/2016         1           0.0           0.0
1          10092      127.0   10/1/2019         1           1.0           0.0
2          10092      127.0   10/1/2019         1           1.0           0.0
85        310039      133.0  11/29/2019         1           0.0           1.0
86        310039      133.0  11/29/2019         1           0.0           1.0
46        180067      142.0    2/8/2020         1           0.0           0.0
47        180067      142.0    2/8/2020         1           0.0           0.0
127       390180      166.0   6/14/2020         1           0.0           0.0
128       390180      166.0   6/14/2020         1           0.0           0.0
129       390180      166.0   6/14/2020         1           0.0           

In [8]:
common = set(threat["CCN"]) & set(medicare["CCN"])
print("\nExample matched CCNs:")
print(sorted(list(common))[:20])

print("\nExample THREAT CCNs:")
print(threat["CCN"].drop_duplicates().head(20).tolist())

print("\nExample Medicare CCNs:")
print(medicare["CCN"].drop_duplicates().head(20).tolist())


Example matched CCNs:
['010001', '010005', '010006', '010007', '010008', '010011', '010012', '010016', '010019', '010021', '010022', '010023', '010024', '010029', '010033', '010034', '010035', '010036', '010039', '010040']

Example THREAT CCNs:
['010045', '010092', '010144', '050039', '050063', '050077', '050204', '050324', '050329', '050424', '050503', '050586', '050696', '050701', '050709', '050757', '050775', '060020', '061307', '061312']

Example Medicare CCNs:
['010001', '010005', '010006', '010007', '010008', '010011', '010012', '010016', '010018', '010019', '010021', '010022', '010023', '010024', '010029', '010033', '010034', '010035', '010036', '010039']


In [9]:
medicare_small = medicare[
    [
        "CCN",
        "Rndrng_Prvdr_Org_Name",
        "Tot_Benes",
        "Tot_Dschrgs",
        "Tot_Cvrd_Days",
        "Tot_Days",
        "Bene_Avg_Age",
        "Bene_Avg_Risk_Scre"
    ]
].copy()

hospital_events = threat.merge(
    medicare_small,
    on="CCN",
    how="left",
    validate="many_to_one"
)

print("Merged shape:", hospital_events.shape)

print(
    hospital_events[
        ["CCN", "Rndrng_Prvdr_Org_Name",
         "Tot_Benes", "Tot_Dschrgs"]
    ].head()
)

print("\nUnmatched Medicare information:")
print(
    hospital_events["Rndrng_Prvdr_Org_Name"]
    .isna()
    .sum()
)

Merged shape: (4349, 22)
      CCN         Rndrng_Prvdr_Org_Name  Tot_Benes  Tot_Dschrgs
0  010045        Fayette Medical Center      159.0        197.0
1  010092   Dch Regional Medical Center     3199.0       4704.0
2  010092   Dch Regional Medical Center     3199.0       4704.0
3  010144  Springhill Memorial Hospital     1340.0       1760.0
4  050039                  Enloe Health     5112.0       7674.0

Unmatched Medicare information:
1546


In [10]:
medicare_small = medicare[
    [
        "CCN",
        "Rndrng_Prvdr_Org_Name",
        "Tot_Benes",
        "Tot_Dschrgs",
        "Tot_Cvrd_Days",
        "Tot_Days",
        "Bene_Avg_Age",
        "Bene_Avg_Risk_Scre"
    ]
].copy()

hospital_events = threat.merge(
    medicare_small,
    on="CCN",
    how="left",
    validate="many_to_one"
)

print("Merged shape:", hospital_events.shape)

print("\nFirst 5 merged rows:")
print(
    hospital_events[
        [
            "CCN",
            "Rndrng_Prvdr_Org_Name",
            "Tot_Benes",
            "Tot_Dschrgs"
        ]
    ].head()
)

print("\nHospitals without Medicare information:",
      hospital_events["Rndrng_Prvdr_Org_Name"].isna().sum())

Merged shape: (4349, 22)

First 5 merged rows:
      CCN         Rndrng_Prvdr_Org_Name  Tot_Benes  Tot_Dschrgs
0  010045        Fayette Medical Center      159.0        197.0
1  010092   Dch Regional Medical Center     3199.0       4704.0
2  010092   Dch Regional Medical Center     3199.0       4704.0
3  010144  Springhill Memorial Hospital     1340.0       1760.0
4  050039                  Enloe Health     5112.0       7674.0

Hospitals without Medicare information: 1546


In [11]:
unmatched = hospital_events[
    hospital_events["Rndrng_Prvdr_Org_Name"].isna()
].copy()

print("Unmatched THREAT rows:", len(unmatched))
print("Unique unmatched CCNs:", unmatched["CCN"].nunique())

print("\nFirst 30 unmatched CCNs:")
print(
    unmatched["CCN"]
    .drop_duplicates()
    .head(30)
    .tolist()
)

Unmatched THREAT rows: 1546
Unique unmatched CCNs: 1546

First 30 unmatched CCNs:
['050757', '061307', '061312', '140143', '261305', '261324', '290022', '291311', '320038', '330223', '331315', '361319', '371324', '390081', '450803', '501319', '510039', '511304', '010010', '010038', '010110', '010138', '011300', '011302', '011304', '011305', '011306', '021301', '021302', '021304']


In [12]:
print("Total THREAT rows:", len(threat))
print("Matched rows:", hospital_events["Rndrng_Prvdr_Org_Name"].notna().sum())
print("Unmatched rows:", hospital_events["Rndrng_Prvdr_Org_Name"].isna().sum())

print("\nUnique THREAT hospitals:", threat["CCN"].nunique())
print(
    "Unique matched hospitals:",
    hospital_events.loc[
        hospital_events["Rndrng_Prvdr_Org_Name"].notna(),
        "CCN"
    ].nunique()
)
print(
    "Unique unmatched hospitals:",
    hospital_events.loc[
        hospital_events["Rndrng_Prvdr_Org_Name"].isna(),
        "CCN"
    ].nunique()
)

Total THREAT rows: 4349
Matched rows: 2803
Unmatched rows: 1546

Unique THREAT hospitals: 4337
Unique matched hospitals: 2791
Unique unmatched hospitals: 1546


In [13]:
threat["attack_date"] = pd.to_datetime(
    threat["attack_date"],
    errors="coerce"
)

hospital_events["attack_year"] = (
    pd.to_datetime(
        hospital_events["attack_date"],
        errors="coerce"
    ).dt.year
)

print(
    hospital_events
    .assign(
        matched=hospital_events["Rndrng_Prvdr_Org_Name"].notna()
    )
    .groupby(["attack_year", "matched"])
    .size()
    .unstack(fill_value=0)
)

matched      False  True 
attack_year              
2016.0           2     22
2017.0           0      3
2018.0           2      2
2019.0           1     25
2020.0           7     51
2021.0           6     39


In [14]:
print("Missing attack dates:",
      threat["attack_date"].isna().sum())

print("\nAttack date examples:")
print(threat["attack_date"].head(20))

print("\nAttack date data type:")
print(threat["attack_date"].dtype)

print("\nOriginal attack_date values:")
print(
    threat["attack_date"]
    .astype(str)
    .head(20)
    .tolist()
)

Missing attack dates: 4189

Attack date examples:
0    2019-10-01
1    2019-10-01
2    2019-10-01
3    2019-07-09
4    2020-01-02
5    2016-02-05
6    2021-05-01
7    2021-05-01
8    2020-09-27
9    2021-05-01
10   2020-09-27
11   2021-05-01
12   2021-05-01
13   2016-03-18
14   2016-08-01
15   2020-09-27
16   2020-09-27
17   2016-03-18
18   2016-03-31
19   2020-09-27
Name: attack_date, dtype: datetime64[us]

Attack date data type:
datetime64[us]

Original attack_date values:
['2019-10-01', '2019-10-01', '2019-10-01', '2019-07-09', '2020-01-02', '2016-02-05', '2021-05-01', '2021-05-01', '2020-09-27', '2021-05-01', '2020-09-27', '2021-05-01', '2021-05-01', '2016-03-18', '2016-08-01', '2020-09-27', '2020-09-27', '2016-03-18', '2016-03-31', '2020-09-27']


In [15]:
print("\nAttack year including missing:")
print(
    threat["attack_date"]
    .dt.year
    .value_counts(dropna=False)
    .sort_index()
)


Attack year including missing:
attack_date
2016.0      24
2017.0       3
2018.0       4
2019.0      26
2020.0      58
2021.0      45
NaN       4189
Name: count, dtype: int64


In [17]:
import pandas as pd

# Reload the original THREAT dataset
threat = pd.read_csv("threat_database.csv")

# Keep the original values
threat["attack_date_raw"] = threat["attack_date"]

# Convert the dates
threat["attack_date"] = pd.to_datetime(
    threat["attack_date_raw"],
    errors="coerce"
)

print("Total rows:", len(threat))
print("Valid attack dates:", threat["attack_date"].notna().sum())
print("Missing attack dates:", threat["attack_date"].isna().sum())

print("\nAttack year distribution:")
print(
    threat["attack_date"]
    .dt.year
    .value_counts(dropna=False)
    .sort_index()
)

Total rows: 4349
Valid attack dates: 160
Missing attack dates: 4189

Attack year distribution:
attack_date
2016.0      24
2017.0       3
2018.0       4
2019.0      26
2020.0      58
2021.0      45
NaN       4189
Name: count, dtype: int64


In [18]:
hospital_events["attack_date"] = pd.to_datetime(
    hospital_events["attack_date"],
    errors="coerce"
)

hospital_events["attack_year"] = (
    hospital_events["attack_date"].dt.year
)

hospital_events["matched"] = (
    hospital_events["Rndrng_Prvdr_Org_Name"].notna()
)

year_match = (
    hospital_events
    .groupby(["attack_year", "matched"])
    .size()
    .unstack(fill_value=0)
)

print(year_match)

matched      False  True 
attack_year              
2016.0           2     22
2017.0           0      3
2018.0           2      2
2019.0           1     25
2020.0           7     51
2021.0           6     39


In [19]:
# Check what the rows without attack dates contain

missing_dates = threat[threat["attack_date"].isna()].copy()

print("Rows with missing attack dates:", len(missing_dates))

print("\nAttacked values:")
print(missing_dates["attacked"].value_counts(dropna=False))

print("\nThreat IDs:")
print(
    "Unique threat IDs:",
    missing_dates["threat_id"].nunique()
)

print("\nMedicare IDs:")
print(
    "Unique Medicare IDs:",
    missing_dates["medicare_id"].nunique()
)

print("\nFirst 20 rows with missing dates:")
print(
    missing_dates[
        [
            "medicare_id",
            "attacked",
            "attack_date",
            "er_diversion",
            "cancel_delay",
            "threat_id",
            "nearest_neighbor"
        ]
    ].head(20)
)

Rows with missing attack dates: 4189

Attacked values:
attacked
0    4189
Name: count, dtype: int64

Threat IDs:
Unique threat IDs: 0

Medicare IDs:
Unique Medicare IDs: 4189

First 20 rows with missing dates:
     medicare_id  attacked attack_date  er_diversion  cancel_delay  threat_id  \
160        10001         0         NaT           NaN           NaN        NaN   
161        10005         0         NaT           NaN           NaN        NaN   
162        10006         0         NaT           NaN           NaN        NaN   
163        10007         0         NaT           NaN           NaN        NaN   
164        10008         0         NaT           NaN           NaN        NaN   
165        10010         0         NaT           NaN           NaN        NaN   
166        10011         0         NaT           NaN           NaN        NaN   
167        10012         0         NaT           NaN           NaN        NaN   
168        10016         0         NaT           NaN         

In [20]:
print("Attacked vs non-attacked:")
print(threat["attacked"].value_counts())

print("\nAttack dates by attacked status:")
print(
    threat.groupby("attacked")["attack_date"]
    .apply(lambda x: x.notna().sum())
)

print("\nThreat IDs by attacked status:")
print(
    threat.groupby("attacked")["threat_id"]
    .apply(lambda x: x.notna().sum())
)

Attacked vs non-attacked:
attacked
0    4189
1     160
Name: count, dtype: int64

Attack dates by attacked status:
attacked
0      0
1    160
Name: attack_date, dtype: int64

Threat IDs by attacked status:
attacked
0      0
1    160
Name: threat_id, dtype: int64


In [21]:
attacked = threat[threat["attacked"] == 1].copy()

print("Attacked rows:", len(attacked))

print(
    "Unique attacked hospitals:",
    attacked["medicare_id"].nunique()
)

print(
    "Unique attack events:",
    attacked["threat_id"].nunique()
)

print("\nHospitals per threat:")
print(
    attacked.groupby("threat_id")["medicare_id"]
    .nunique()
    .sort_values(ascending=False)
    .head(20)
)

Attacked rows: 160
Unique attacked hospitals: 148
Unique attack events: 74

Hospitals per threat:
threat_id
188.0    22
133.0    11
7.0       9
263.0     9
264.0     8
186.0     4
248.0     4
198.0     3
28.0      3
273.0     3
254.0     2
135.0     2
192.0     2
259.0     2
166.0     2
127.0     2
90.0      2
54.0      2
4.0       2
11.0      1
Name: medicare_id, dtype: int64


In [22]:
# Compare Medicare matching separately for attacked and non-attacked hospitals

hospital_events["matched_medicare"] = (
    hospital_events["Rndrng_Prvdr_Org_Name"].notna()
)

print("Medicare matching by attack status:")
print(
    hospital_events
    .groupby("attacked")["matched_medicare"]
    .value_counts()
    .unstack(fill_value=0)
)

print("\nUnique hospitals by attack status:")

for status in [0, 1]:
    subset = hospital_events[hospital_events["attacked"] == status]

    print(
        f"attacked={status}:",
        subset["CCN"].nunique(),
        "unique hospitals"
    )

    print(
        "  matched:",
        subset.loc[
            subset["matched_medicare"], "CCN"
        ].nunique()
    )

    print(
        "  unmatched:",
        subset.loc[
            ~subset["matched_medicare"], "CCN"
        ].nunique()
    )

Medicare matching by attack status:
matched_medicare  False  True 
attacked                      
0                  1528   2661
1                    18    142

Unique hospitals by attack status:
attacked=0: 4189 unique hospitals
  matched: 2661
  unmatched: 1528
attacked=1: 148 unique hospitals
  matched: 130
  unmatched: 18


In [23]:
# Show the 18 attacked hospitals that did not match
unmatched_attacked = (
    hospital_events[
        (hospital_events["attacked"] == 1) &
        (~hospital_events["matched_medicare"])
    ]
    .drop_duplicates(subset=["CCN"])
)

print("Unmatched attacked hospitals:", len(unmatched_attacked))

print(
    unmatched_attacked[
        [
            "CCN",
            "medicare_id",
            "attack_date",
            "threat_id",
            "er_diversion",
            "cancel_delay"
        ]
    ].sort_values("attack_date")
)

Unmatched attacked hospitals: 18
        CCN  medicare_id attack_date  threat_id  er_diversion  cancel_delay
155  501319       501319  2016-01-01        6.0           0.0           0.0
18   050757        50757  2016-03-31        8.0           0.0           0.0
66   261324       261324  2018-07-09       80.0           1.0           0.0
157  510039       510039  2018-11-23       90.0           1.0           0.0
22   061312        61312  2019-06-02      339.0           0.0           0.0
21   061307        61307  2020-04-09      150.0           0.0           0.0
126  390081       390081  2020-06-14      166.0           0.0           0.0
152  450803       450803  2020-07-25      174.0           0.0           0.0
74   290022       290022  2020-09-27      188.0           1.0           1.0
81   291311       291311  2020-09-27      188.0           1.0           1.0
100  330223       330223  2020-10-27      198.0           1.0           0.0
101  331315       331315  2020-10-27      198.0        

In [24]:
# Check whether these CCNs look valid
print("\nCCN values:")
print(
    unmatched_attacked["CCN"]
    .astype(str)
    .tolist()
)


CCN values:
['050757', '061307', '061312', '140143', '261305', '261324', '290022', '291311', '320038', '330223', '331315', '361319', '371324', '390081', '450803', '501319', '510039', '511304']


In [25]:
# Check Medicare variables for attacked hospitals

attacked_matched = hospital_events[
    (hospital_events["attacked"] == 1) &
    (hospital_events["matched_medicare"])
].copy()

print("Matched attacked hospital observations:",
      len(attacked_matched))

print("\nMissing Medicare information:")
print(
    attacked_matched[
        [
            "Tot_Benes",
            "Tot_Dschrgs",
            "Tot_Cvrd_Days",
            "Tot_Days",
            "Bene_Avg_Age",
            "Bene_Avg_Risk_Scre"
        ]
    ].isna().sum()
)

print("\nSummary of Medicare variables:")
print(
    attacked_matched[
        [
            "Tot_Benes",
            "Tot_Dschrgs",
            "Tot_Cvrd_Days",
            "Tot_Days",
            "Bene_Avg_Age",
            "Bene_Avg_Risk_Scre"
        ]
    ].describe()
)

Matched attacked hospital observations: 142

Missing Medicare information:
Tot_Benes             0
Tot_Dschrgs           0
Tot_Cvrd_Days         0
Tot_Days              0
Bene_Avg_Age          0
Bene_Avg_Risk_Scre    0
dtype: int64

Summary of Medicare variables:
          Tot_Benes   Tot_Dschrgs  Tot_Cvrd_Days       Tot_Days  Bene_Avg_Age  \
count    142.000000    142.000000     142.000000     142.000000    142.000000   
mean    2137.781690   2875.528169   15864.669014   16583.119718     75.401402   
std     1758.389544   2415.717798   16163.121379   17081.679200      2.526590   
min       12.000000     12.000000      60.000000      60.000000     61.117541   
25%      920.500000   1267.750000    6061.250000    6211.750000     74.252019   
50%     1742.000000   2333.500000   12104.000000   12365.500000     75.726649   
75%     2728.500000   3622.250000   19547.250000   20523.250000     76.925385   
max    11235.000000  15416.000000  117501.000000  122301.000000     80.898773   

      

In [26]:
# ==========================================
# CREATE CLEAN ANALYSIS DATA
# ==========================================

analysis_data = hospital_events[
    hospital_events["matched_medicare"] == True
].copy()

print("Analysis dataset shape:", analysis_data.shape)

print("\nAttack status:")
print(
    analysis_data["attacked"].value_counts()
)

print("\nUnique hospitals:")
print(
    analysis_data["CCN"].nunique()
)

print("\nUnique attack events:")
print(
    analysis_data.loc[
        analysis_data["attacked"] == 1,
        "threat_id"
    ].nunique()
)

Analysis dataset shape: (2803, 25)

Attack status:
attacked
0    2661
1     142
Name: count, dtype: int64

Unique hospitals:
2791

Unique attack events:
64


In [1]:
# ==========================================
# STEP 5 — CHECK HOSPITAL DUPLICATION
# ==========================================

hospital_counts = (
    analysis_data
    .groupby("CCN")
    .size()
    .sort_values(ascending=False)
)

print("Hospitals with more than one observation:")
print(
    (hospital_counts > 1).sum()
)

print("\nMaximum observations for one hospital:")
print(
    hospital_counts.max()
)

print("\nObservation count distribution:")
print(
    hospital_counts.value_counts().sort_index()
)

NameError: name 'analysis_data' is not defined